Prepared by: Mahboob Attar  
Date: 12 February 2026

## Assignment Objective
Clean messy member signup data and produce:
- `members_final.csv` → Clean CRM-ready file
- `quarantine.csv` → Low quality / test leads

## Business Rules
1. Standardize all dates to YYYY-MM-DD
2. Remove duplicate members
3. Move "low quality" leads to quarantine
4. If user signed up for both Plan A and Plan B:
   - Keep most recent signup
   - Add flag `is_multi_plan = True`


## 1. Import Libraries

In [19]:
import pandas as pd
import numpy as np

# Date handling
from datetime import datetime

# For detecting patterns (low quality detection)
import re


## 2. Load Data and Inspect

The file is a  Excel binary file (.xls)
Not a CSV disguised as .xls

In [49]:
df = pd.read_excel("signup.xls", engine="xlrd")

print("Shape of dataset:", df.shape)
print("\nColumn Names:\n", df.columns)

print("\nData Types:\n", df.dtypes)

print("\nMissing Values:")
print(df.isna().sum())
df.head(4)

Shape of dataset: (8, 1)

Column Names:
 Index(['name,email,signup_date,plan,notes'], dtype='str')

Data Types:
 name,email,signup_date,plan,notes    str
dtype: object

Missing Values:
name,email,signup_date,plan,notes    0
dtype: int64


,"name,email,signup_date,plan,notes"
0,"John Doe,john.doe@gmail.com,2023-01-15,Plan A,"
1,"Jane Smith,jane.smith@yahoo.com,02/12/23,Plan ..."
2,"Doe, John,john.doe@gmail.com,2023-01-20,Plan B..."
3,"Test User,test@test.com,2023-05-01,Plan A,IGNO..."


## 3. Handling Structural Data Issues (Comma Inside Names)

### Problem Identified
The Excel file contains only one column, where values are comma-separated.
However, some names contain commas (e.g., "Doe, John"), which caused incorrect splitting.

Example issue:
Doe, John,john.doe@gmail.com,2023-01-20,Plan B,Phone signup

When splitting by comma, this creates 6 columns instead of 5.


In [50]:
# Split everything first
df_split = df.iloc[:, 0].str.split(",", expand=True)

print("Columns after split:", df_split.shape[1])
df_split.head(4)


Columns after split: 6


,0,1,2,3,4,5
0,John Doe,john.doe@gmail.com,2023-01-15,Plan A,,NaN
1,Jane Smith,jane.smith@yahoo.com,02/12/23,Plan B,Interested in keto,NaN
2,Doe,John,john.doe@gmail.com,2023-01-20,Plan B,Phone signup
3,Test User,test@test.com,2023-05-01,Plan A,IGNORE THIS,NaN


### 5. Solution Strategy
Instead of blindly renaming columns, we:

1. Split the row by comma.
2. Detect whether the email column shifted by checking for "@".
3. If shifted:
   - Combine first two columns to reconstruct the full name.
4. Otherwise:
   - Use normal column positions.
5. Rebuild a clean DataFrame row-by-row.

This ensures:
- All rows are preserved
- Names containing commas are handled correctly
- No assumptions are made about perfect data formatting


In [51]:
clean_rows = []

for _, row in df_split.iterrows():
    
    # If column 2 contains "@", name was split
    if pd.notna(row[2]) and "@" in str(row[2]):
        name = f"{row[0]}, {row[1]}"
        email = row[2]
        signup_date = row[3]
        plan = row[4]
        notes = row[5]
    else:
        name = row[0]
        email = row[1]
        signup_date = row[2]
        plan = row[3]
        notes = row[4]
    
    clean_rows.append([name, email, signup_date, plan, notes])

df_clean = pd.DataFrame(clean_rows, 
                        columns=["name", "email", "signup_date", "plan", "notes"])

# Strip whitespace
df_clean = df_clean.apply(lambda x: x.str.strip())

df_clean


,name,email,signup_date,plan,notes
0,John Doe,john.doe@gmail.com,2023-01-15,Plan A,
1,Jane Smith,jane.smith@yahoo.com,02/12/23,Plan B,Interested in keto
2,"Doe, John",john.doe@gmail.com,2023-01-20,Plan B,Phone signup
3,Test User,test@test.com,2023-05-01,Plan A,IGNORE THIS
4,J4ne Smith,jane.smith@yahoo.com,2023-12-02,Plan B,
5,Bob,bob@outlook.com,Yesterday,Plan A,
6,Alice Wonderland,alice@wonder.land,2023-07-07,Plan C,VIP
7,John Doe,j.doe+filter@gmail.com,2023-01-22,Plan A,


## 6. Data check before aply business logic

In [52]:
print("Shape:", df_clean.shape)
print("\nUnique Plans:", df_clean["plan"].unique())
print("\nSample Dates:", df_clean["signup_date"].unique())
print("\nEmails:")
print(df_clean["email"])

Shape: (8, 5)

Unique Plans: <StringArray>
['Plan A', 'Plan B', 'Plan C']
Length: 3, dtype: str

Sample Dates: <StringArray>
['2023-01-15',   '02/12/23', '2023-01-20', '2023-05-01', '2023-12-02',
  'Yesterday', '2023-07-07', '2023-01-22']
Length: 8, dtype: str

Emails:
0        john.doe@gmail.com
1      jane.smith@yahoo.com
2        john.doe@gmail.com
3             test@test.com
4      jane.smith@yahoo.com
5           bob@outlook.com
6         alice@wonder.land
7    j.doe+filter@gmail.com
Name: email, dtype: str


## 7. Standardizing Signup Dates

### Problem
Signup dates appear in multiple formats:
- YYYY-MM-DD
- MM/DD/YY
- Natural language (e.g., "Yesterday")

### Business Requirement
All dates must be standardized to:
YYYY-MM-DD format.

### Approach
1. Convert all dates using pandas `to_datetime()`
2. Handle invalid formats safely using `errors='coerce'`
3. Convert final result to YYYY-MM-DD string format


In [64]:
# Convert signup_date to datetime
df_clean["signup_date"] = pd.to_datetime(
    df_clean["signup_date"],
    errors="coerce"
)

# Convert back to standard format
df_clean["signup_date"] = df_clean["signup_date"].dt.strftime("%Y-%m-%d")

df_clean


,name,email,signup_date,plan,notes
0,John Doe,john.doe@gmail.com,2023-01-15,Plan A,
1,Jane Smith,jane.smith@yahoo.com,NaN,Plan B,Interested in keto
2,"Doe, John",john.doe@gmail.com,2023-01-20,Plan B,Phone signup
3,Test User,test@test.com,2023-05-01,Plan A,IGNORE THIS
4,J4ne Smith,jane.smith@yahoo.com,2023-12-02,Plan B,
5,Bob,bob@outlook.com,NaN,Plan A,
6,Alice Wonderland,alice@wonder.land,2023-07-07,Plan C,VIP
7,John Doe,j.doe+filter@gmail.com,2023-01-22,Plan A,


## (Improvement) — Smarter Date Parsing

We want:

02/12/23 → parsed correctly

"Yesterday" → converted to actual date

Invalid values → quarantined

In [63]:
# Replace "Yesterday" before datetime conversion
df_clean["signup_date"] = df_clean["signup_date"].replace(
    "Yesterday",
    (pd.Timestamp.today() - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
)

# Convert all to datetime
df_clean["signup_date"] = pd.to_datetime(
    df_clean["signup_date"],
    errors="coerce"
)

# Convert to standard format
df_clean["signup_date"] = df_clean["signup_date"].dt.strftime("%Y-%m-%d")

df_clean


,name,email,signup_date,plan,notes
0,John Doe,john.doe@gmail.com,2023-01-15,Plan A,
1,Jane Smith,jane.smith@yahoo.com,NaN,Plan B,Interested in keto
2,"Doe, John",john.doe@gmail.com,2023-01-20,Plan B,Phone signup
3,Test User,test@test.com,2023-05-01,Plan A,IGNORE THIS
4,J4ne Smith,jane.smith@yahoo.com,2023-12-02,Plan B,
5,Bob,bob@outlook.com,NaN,Plan A,
6,Alice Wonderland,alice@wonder.land,2023-07-07,Plan C,VIP
7,John Doe,j.doe+filter@gmail.com,2023-01-22,Plan A,


In [94]:
clean_rows = []

for _, row in df_split.iterrows():
    
    if pd.notna(row[2]) and "@" in str(row[2]):
        name = f"{row[0]}, {row[1]}"
        email = row[2]
        signup_date = row[3]
        plan = row[4]
        notes = row[5]
    else:
        name = row[0]
        email = row[1]
        signup_date = row[2]
        plan = row[3]
        notes = row[4]
    
    clean_rows.append([name, email, signup_date, plan, notes])

df_clean = pd.DataFrame(
    clean_rows,
    columns=["name", "email", "signup_date", "plan", "notes"]
)

df_clean = df_clean.apply(lambda x: x.str.strip())

df_clean


,name,email,signup_date,plan,notes
0,John Doe,john.doe@gmail.com,2023-01-15,Plan A,
1,Jane Smith,jane.smith@yahoo.com,02/12/23,Plan B,Interested in keto
2,"Doe, John",john.doe@gmail.com,2023-01-20,Plan B,Phone signup
3,Test User,test@test.com,2023-05-01,Plan A,IGNORE THIS
4,J4ne Smith,jane.smith@yahoo.com,2023-12-02,Plan B,
5,Bob,bob@outlook.com,Yesterday,Plan A,
6,Alice Wonderland,alice@wonder.land,2023-07-07,Plan C,VIP
7,John Doe,j.doe+filter@gmail.com,2023-01-22,Plan A,


In [100]:
# Handle 'Yesterday' before conversion
df_clean["signup_date"] = df_clean["signup_date"].replace(
    "Yesterday",
    (pd.Timestamp.today() - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
)

# Convert with automatic format detection
df_clean["signup_date"] = pd.to_datetime(
    df_clean["signup_date"],
    errors="coerce",
    dayfirst=False
)

# Standard YYYY-MM-DD
df_clean["signup_date"] = df_clean["signup_date"].dt.strftime("%Y-%m-%d")

df_clean


,name,email,signup_date,plan,notes
0,John Doe,john.doe@gmail.com,2023-01-15,Plan A,
1,Jane Smith,jane.smith@yahoo.com,NaN,Plan B,Interested in keto
2,"Doe, John",john.doe@gmail.com,2023-01-20,Plan B,Phone signup
3,Test User,test@test.com,2023-05-01,Plan A,IGNORE THIS
4,J4ne Smith,jane.smith@yahoo.com,2023-12-02,Plan B,
5,Bob,bob@outlook.com,2026-02-11,Plan A,
6,Alice Wonderland,alice@wonder.land,2023-07-07,Plan C,VIP
7,John Doe,j.doe+filter@gmail.com,2023-01-22,Plan A,


## We need to confirm:

Is it really "02/12/23"?

Or did it become blank?

Or does it contain whitespace?

Because if the raw value is already lost, parsing cannot recover it.

In [87]:
df_clean.loc[1, "signup_date"]
row = df_split.iloc[1]
print(row)

0              Jane Smith
1    jane.smith@yahoo.com
2                02/12/23
3                  Plan B
4      Interested in keto
5                     NaN
Name: 1, dtype: str


In [101]:
clean_rows = []

for _, row in df_split.iterrows():
    
    if pd.notna(row[2]) and "@" in str(row[2]):
        name = f"{row[0]}, {row[1]}"
        email = row[2]
        signup_date = row[3]
        plan = row[4]
        notes = row[5]
    else:
        name = row[0]
        email = row[1]
        signup_date = row[2]
        plan = row[3]
        notes = row[4]
    
    clean_rows.append([name, email, signup_date, plan, notes])

df_clean = pd.DataFrame(
    clean_rows,
    columns=["name", "email", "signup_date", "plan", "notes"]
)

df_clean = df_clean.apply(lambda x: x.str.strip())

df_clean.head(8)


,name,email,signup_date,plan,notes
0,John Doe,john.doe@gmail.com,2023-01-15,Plan A,
1,Jane Smith,jane.smith@yahoo.com,02/12/23,Plan B,Interested in keto
2,"Doe, John",john.doe@gmail.com,2023-01-20,Plan B,Phone signup
3,Test User,test@test.com,2023-05-01,Plan A,IGNORE THIS
4,J4ne Smith,jane.smith@yahoo.com,2023-12-02,Plan B,
5,Bob,bob@outlook.com,Yesterday,Plan A,
6,Alice Wonderland,alice@wonder.land,2023-07-07,Plan C,VIP
7,John Doe,j.doe+filter@gmail.com,2023-01-22,Plan A,


In [102]:
def parse_date(value):
    if pd.isna(value):
        return pd.NaT
    
    value = value.strip()
    
    if value.lower() == "yesterday":
        return pd.Timestamp.today() - pd.Timedelta(days=1)
    
    # Try ISO first
    try:
        return pd.to_datetime(value, format="%Y-%m-%d")
    except:
        pass
    
    # Try MM/DD/YY
    try:
        return pd.to_datetime(value, format="%m/%d/%y")
    except:
        return pd.NaT


df_clean["signup_date"] = df_clean["signup_date"].apply(parse_date)

df_clean["signup_date"] = df_clean["signup_date"].dt.strftime("%Y-%m-%d")

df_clean


,name,email,signup_date,plan,notes
0,John Doe,john.doe@gmail.com,2023-01-15,Plan A,
1,Jane Smith,jane.smith@yahoo.com,2023-02-12,Plan B,Interested in keto
2,"Doe, John",john.doe@gmail.com,2023-01-20,Plan B,Phone signup
3,Test User,test@test.com,2023-05-01,Plan A,IGNORE THIS
4,J4ne Smith,jane.smith@yahoo.com,2023-12-02,Plan B,
5,Bob,bob@outlook.com,2026-02-11,Plan A,
6,Alice Wonderland,alice@wonder.land,2023-07-07,Plan C,VIP
7,John Doe,j.doe+filter@gmail.com,2023-01-22,Plan A,


Immediately Parse Dates Correctly,Date Standardization is DONE

## 8. Identifying Low Quality Leads (Quarantine Logic)

Not all records represent real customers.

We flag leads as low-quality if they:

1. Contain obvious test indicators in name or email
2. Have notes like "IGNORE THIS"
3. Are missing critical fields (name, email, signup_date)

Instead of deleting them, we move them to `quarantine.csv`
to protect CRM data quality while keeping records for audit.


In [104]:
# Define quarantine rules

low_quality_mask = (
    df_clean["name"].str.contains("test", case=False, na=False) |
    df_clean["email"].str.contains("test", case=False, na=False) |
    df_clean["notes"].str.contains("ignore", case=False, na=False) |
    df_clean["name"].isna() |
    df_clean["email"].isna() |
    df_clean["signup_date"].isna()
)

# Create quarantine dataset
quarantine_df = df_clean[low_quality_mask].copy()

# Remaining clean dataset
df_valid = df_clean[~low_quality_mask].copy()

print("Total rows:", len(df_clean))
print("Quarantined rows:", len(quarantine_df))
print("Valid rows:", len(df_valid))

quarantine_df


Total rows: 8
Quarantined rows: 1
Valid rows: 7


,name,email,signup_date,plan,notes
3,Test User,test@test.com,2023-05-01,Plan A,IGNORE THIS


## 9. Deduplication & Multi-Plan Handling

### Business Rule
If a user signed up multiple times:

- Keep only ONE record in final dataset.

If they signed up for multiple plans:
- Keep the most recent signup.
- Add a flag: is_multi_plan = True.

If they signed up multiple times for the same plan:
- Keep the most recent record.
- is_multi_plan = False.


### 9.1 Sort by Most Recent Date

In [ ]:
# Convert signup_date back to datetime for sorting
df_valid["signup_date"] = pd.to_datetime(df_valid["signup_date"])

df_valid = df_valid.sort_values(
    by=["email", "signup_date"],
    ascending=[True, False]
)

df_valid

,name,email,signup_date,plan,notes,is_multi_plan
6,Alice Wonderland,alice@wonder.land,2023-07-07,Plan C,VIP,False
5,Bob,bob@outlook.com,2026-02-11,Plan A,,False
7,John Doe,j.doe+filter@gmail.com,2023-01-22,Plan A,,False
4,J4ne Smith,jane.smith@yahoo.com,2023-12-02,Plan B,,False
1,Jane Smith,jane.smith@yahoo.com,2023-02-12,Plan B,Interested in keto,False
2,"Doe, John",john.doe@gmail.com,2023-01-20,Plan B,Phone signup,True
0,John Doe,john.doe@gmail.com,2023-01-15,Plan A,,True


### 9.2 Detect Multi-Plan Users

In [ ]:
plan_counts = df_valid.groupby("email")["plan"].nunique()

df_valid["is_multi_plan"] = df_valid["email"].map(plan_counts) > 1

df_valid


,name,email,signup_date,plan,notes,is_multi_plan
6,Alice Wonderland,alice@wonder.land,2023-07-07,Plan C,VIP,False
5,Bob,bob@outlook.com,2026-02-11,Plan A,,False
7,John Doe,j.doe+filter@gmail.com,2023-01-22,Plan A,,False
4,J4ne Smith,jane.smith@yahoo.com,2023-12-02,Plan B,,False
1,Jane Smith,jane.smith@yahoo.com,2023-02-12,Plan B,Interested in keto,False
2,"Doe, John",john.doe@gmail.com,2023-01-20,Plan B,Phone signup,True
0,John Doe,john.doe@gmail.com,2023-01-15,Plan A,,True


### 9.3 Keep Only Most Recent Record Per Email

In [121]:
df_final = df_valid.drop_duplicates(subset="email", keep="first")

df_final


,name,email,signup_date,plan,notes,is_multi_plan
6,Alice Wonderland,alice@wonder.land,2023-07-07,Plan C,VIP,False
5,Bob,bob@outlook.com,2026-02-11,Plan A,,False
7,John Doe,j.doe+filter@gmail.com,2023-01-22,Plan A,,False
4,J4ne Smith,jane.smith@yahoo.com,2023-12-02,Plan B,,False
2,"Doe, John",john.doe@gmail.com,2023-01-20,Plan B,Phone signup,True


## 10. Exporting Final Outputs

We generate two output files:

1. members_final.csv → Clean, deduplicated, CRM-ready data
2. quarantine.csv → Low-quality or suspicious leads

This ensures:
- CRM only receives valid records
- Suspicious data is preserved for audit/review


In [122]:
# Convert signup_date back to string format
df_final["signup_date"] = df_final["signup_date"].dt.strftime("%Y-%m-%d")

# Export files
df_final.to_csv("members_final.csv", index=False)
quarantine_df.to_csv("quarantine.csv", index=False)

print("Files exported successfully.")
print("Final clean records:", len(df_final))
print("Quarantined records:", len(quarantine_df))


Files exported successfully.
Final clean records: 5
Quarantined records: 1


# Assignment Questions

## 1. What logic did you apply?

I followed a structured ETL (Extract, Transform, Load) approach:

1. Fixed structural issues caused by comma-separated values inside Excel.
2. Reconstructed proper columns safely.
3. Standardized signup dates to YYYY-MM-DD format, handling multiple formats and natural language values.
4. Identified low-quality leads based on business rules (test accounts, missing data, ignore notes).
5. Used email as a unique identifier to deduplicate records.
6. Applied business logic to detect multi-plan users and kept only the most recent signup.

This ensured the final dataset is clean, consistent, and CRM-ready.

---

## 2. What was most frustrating about the data, and why delete 20% of leads?

The most frustrating issue was structural inconsistency:
- Names containing commas broke column alignment.
- Multiple date formats required explicit handling.
- Test records were mixed with real leads.

To a non-technical manager, I would explain:

"We did not randomly delete leads. We removed records that were clearly internal tests, incomplete, or invalid. Keeping them would reduce CRM accuracy, mislead performance metrics, and waste sales effort. Removing approximately 20% of low-quality data improves conversion tracking and protects business decision-making."
